<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_timo_attention_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 5: Attention V2

**Autor:** Timo · **Datum:** 19.11.2026

Wie schnell trainiert unser GPT-2 Small, und was bringen die Schritte von Folie 20?
Jeder Schritt wird mit derselben Methode gemessen: Tokens pro Sekunde, Spitzenspeicher und MFU.

| Nr. | Schritt | Modell | Präzision | Attention |
|---|---|---|---|---|
| 1 | Ausgangsstand | `GPTModel` (Woche 4) | fp32 | eigener Code |
| 2 | 16 Bit mit `autocast` | `GPTModel` | bf16, Gewichte fp32 | eigener Code |
| 3 | 16 Bit, ganzes Modell | `GPTModel` | bf16 | eigener Code |
| 4 | torch.nn ohne Flash | `GPTModelV2` | wie Nr. 2 | `scaled_dot_product_attention`, Math-Kernel |
| 5 | torch.nn mit Flash | `GPTModelV2` | wie Nr. 2 | `scaled_dot_product_attention`, Kernel nach Wahl von PyTorch |
| 6 | `torch.compile` | `GPTModelV2` | wie Nr. 2 | wie Nr. 5 |

Entscheidungen und Gründe stehen in [`docs/attention-v2.md`](https://github.com/Timbo3399/INFM_SOTA2/blob/main/docs/attention-v2.md).
Der Code liegt in `src/infm_sota2/` und hat Unittests, dieses Notebook ruft ihn nur auf.

## 0. Einstellungen und Setup

`SMOKE_TEST = True` prüft den Ablauf lokal auf der CPU mit einem winzigen Modell. Die Zahlen haben dann keine Aussagekraft.

Für den echten Lauf eine GPU ab der A100-Generation nehmen, in Colab die L4 oder A100.
Die T4 kann bf16 nicht in Hardware, und der FlashAttention-Kernel von PyTorch braucht mindestens eine A100 (Compute Capability 8.0).
Auf der T4 laufen die 16-Bit-Varianten deshalb mit fp16 und GradScaler, Nr. 3 wird übersprungen.

In [ ]:
SMOKE_TEST = False

In [ ]:
import sys

# Colab installiert das Paket aus diesem Branch. Vor dem Merge nach main zum Testen vorübergehend
# z. B. "week/05" eintragen und vor dem Speichern zurück auf "main" setzen.
BRANCH = "main"
if "google.colab" in sys.modules:
    # Eigenes Paket installieren, Colabs torch/tiktoken/wandb bleiben unangetastet
    %pip install -q --no-deps "git+https://github.com/Timbo3399/INFM_SOTA2.git@{BRANCH}"

import torch

from infm_sota2.env import Environment

env = Environment.detect(smoke_test=SMOKE_TEST)
for warning in env.check_versions():
    print(warning)
print(env.describe())

### So wird gemessen

Vorlage ist das Benchmark-Skript aus Moodle. Unser `Benchmark` misst dieselben Größen, aber:

- **Erst aufwärmen, dann messen.** Die ersten Schritte enthalten CUDA-Initialisierung und bei `torch.compile` das Kompilieren.
- **Ganze Trainingsschritte:** Gradienten löschen, Forward, Loss, Backward und AdamW-Schritt, gemittelt über mehrere Schritte.
- **Alle Varianten mit derselben Batchgröße und Sequenzlänge** und mit zufälligen Tokens. Der Inhalt der Daten ändert die Geschwindigkeit nicht.
- **MFU** = erreichte FLOPs pro Sekunde geteilt durch die Spitzenleistung der GPU.
  FLOPs pro Token = 6 · (Gewichte aller Linear-Schichten) + 12 · Schichten · Sequenzlänge · Embedding (PaLM-Paper, Anhang B).
  Das Moodle-Skript rechnet gegen Spitzenwerte ohne Tensor Cores, seine MFU fällt deshalb etwa viermal zu hoch aus.
  Dass es mit `thop` die Matrixprodukte der Attention nicht mitzählt, drückt sie dagegen um rund 13 % (GPT-2 Small, 1024 Tokens).

In [ ]:
from infm_sota2.model import GPT2_SMALL_CONFIG

if SMOKE_TEST:
    MODEL_CONFIG = {
        **GPT2_SMALL_CONFIG,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 2,
        "n_layers": 2,
    }
    BENCH = {"batch_size": 2, "seq_len": 32, "warmup_steps": 1, "measure_steps": 2, "seed": 123}
    print("SMOKE_TEST: winziges Modell auf der CPU, Zahlen ohne Aussagekraft")
else:
    MODEL_CONFIG = GPT2_SMALL_CONFIG
    BENCH = {"batch_size": 4, "seq_len": 1024, "warmup_steps": 3, "measure_steps": 10, "seed": 123}
print(BENCH)

In [ ]:
from dataclasses import asdict

import pandas as pd

from infm_sota2.benchmark import Benchmark, Variant
from infm_sota2.model import GPTModel, GPTModelV2

bench = Benchmark(MODEL_CONFIG, device=env.device, **BENCH)

# Ohne bf16-Tensor-Cores (T4) bleibt für 16 Bit nur fp16 mit GradScaler
HAS_BF16 = env.device.type != "cuda" or torch.cuda.get_device_capability(env.device)[0] >= 8
MIXED = "bf16-mixed" if HAS_BF16 else "fp16-mixed"
# Im Smoke-Test prüft "eager" nur den Ablauf, "inductor" braucht unter Windows einen C++-Compiler
COMPILE = "eager" if SMOKE_TEST else "inductor"

VARIANTS = [
    Variant("1 Ausgangsstand", GPTModel, "fp32", "auto", None),
    Variant("2 16 Bit mit autocast", GPTModel, MIXED, "auto", None),
    Variant("3 16 Bit, ganzes Modell", GPTModel, "bf16", "auto", None),
    Variant("4 torch.nn ohne Flash", GPTModelV2, MIXED, "math", None),
    Variant("5 torch.nn mit Flash", GPTModelV2, MIXED, "auto", None),
    Variant("6 torch.compile", GPTModelV2, MIXED, "auto", COMPILE),
]
results = {}  # Name -> Ergebnis, eine Zelle erneut auszuführen ersetzt ihre Zeilen


def table() -> pd.DataFrame:
    columns = {
        "variant": "Variante",
        "tokens_per_second": "Tokens/s",
        "ms_per_step": "ms/Schritt",
        "peak_memory_gb": "Speicher (GB)",
        "flops_per_token": "FLOPs/Token",
        "mfu": "MFU",
        "error": "Fehler",
    }
    rows = map(asdict, results.values())
    return pd.DataFrame(rows).rename(columns=columns).set_index("Variante")


def measure(*numbers: int) -> pd.DataFrame:
    """Misst die Varianten mit diesen Nummern und zeigt alle bisherigen Ergebnisse."""
    for number in numbers:
        variant = VARIANTS[number - 1]
        if variant.precision == "bf16" and not HAS_BF16:
            print(f"{variant.name}: übersprungen, die GPU kann bf16 nicht in Hardware")
            continue
        results[variant.name] = bench.run(variant)
    return table()

## 1. Ausgangsstand: unser `GPTModel` in fp32

Das Modell aus Woche 4 (Raschka, Kapitel 4): eigene LayerNorm, eigene Attention mit gespeicherter Maske und einer T×T-Matrix pro Kopf, alles in 32 Bit.
Folie 20, Punkt 1.

In [ ]:
measure(1)

## 2. 16 Bit

bf16 hat denselben Exponenten wie fp32, aber nur 7 statt 23 Bit Mantisse (Folie 11). Es braucht halb so viel Speicher,
und Matrixprodukte in bf16 laufen auf den Tensor Cores (Folie 13). Wir messen zwei Formen:

- **Nr. 2, mit `autocast`** wie auf Folie 12: Die Gewichte bleiben in fp32, gerechnet wird in bf16. So trainiert man üblicherweise.
- **Nr. 3, ganzes Modell in bf16** wie im Moodle-Skript (`model.to(torch.bfloat16)`): spart mehr Speicher,
  aber AdamW rechnet dann auch die Updates in bf16 und verliert kleine Änderungen.

In [ ]:
measure(2, 3)

## 3. GPT aus `torch.nn`-Bausteinen

`GPTModelV2` ist dieselbe Architektur, aber mit den fertigen Bausteinen von PyTorch: `nn.LayerNorm`, `nn.GELU`,
ein gemeinsames `nn.Linear` für Q, K und V und `F.scaled_dot_product_attention(is_causal=True)` statt Maske, Softmax und zwei Matrixprodukten von Hand.

- **Nr. 4** zwingt `scaled_dot_product_attention` auf den Math-Kernel. Der rechnet wie unser eigener Code mit der vollen T×T-Matrix in 16 Bit.
  Von sich aus würde PyTorch im Math-Kernel in fp32 rechnen, deshalb erlaubt der Benchmark hier 16 Bit (`allow_fp16_bf16_reduction_math_sdp`).
- **Nr. 5** lässt PyTorch den Kernel wählen. Ab der A100-Generation ist das FlashAttention (Folie 15):
  Die Attention wird in Kacheln im schnellen On-Chip-Speicher gerechnet, die T×T-Matrix landet nie im GPU-Speicher.

Der Unterschied zwischen Nr. 4 und Nr. 5 ist also allein FlashAttention.

In [ ]:
measure(4, 5)

### Ist `GPTModelV2` dasselbe Modell?

Beide Modelle bekommen die OpenAI-Gewichte und setzen „The Elves are“ ohne Zufall fort.
`GPTModel` rechnet die GELU exakt, `GPTModelV2` wie das Original-GPT-2 mit der tanh-Näherung. Die Logits weichen deshalb leicht ab.
Dass V2 mit umgerechneten Gewichten sonst genau dasselbe rechnet, prüft der Unittest `test_matches_v1_with_converted_weights`.

In [ ]:
import tiktoken

from infm_sota2.model import (
    GPT2_SMALL_WEIGHTS_SHA256,
    GPT2_SMALL_WEIGHTS_URL,
    download_weights,
    load_pretrained,
)
from infm_sota2.training import complete

if SMOKE_TEST:
    print("SMOKE_TEST: Vergleich mit den OpenAI-Gewichten übersprungen")
else:
    weights_path = download_weights(
        GPT2_SMALL_WEIGHTS_URL,
        GPT2_SMALL_WEIGHTS_SHA256,
        env.checkpoint_dir / "gpt2-small-124M.pth",
    )
    tokenizer = tiktoken.get_encoding("gpt2")
    prompt = torch.tensor([tokenizer.encode("The Elves are")], device=env.device)
    logits = {}
    for model_cls in (GPTModel, GPTModelV2):
        model = load_pretrained(model_cls, GPT2_SMALL_CONFIG, weights_path, env.device).eval()
        with torch.no_grad():
            logits[model_cls.__name__] = model(prompt)
        text = complete(
            model, tokenizer, "The Elves are", max_new_tokens=30,
            context_size=GPT2_SMALL_CONFIG["context_length"], temperature=0.0, top_k=None, generator=None,
        )
        print(f"{model_cls.__name__}: The Elves are{text}")
        del model
    difference = (logits["GPTModel"] - logits["GPTModelV2"]).abs().max().item()
    print(f"Größte Abweichung der Logits: {difference:.4f}")

## 4. `torch.compile`

`torch.compile` erfasst den Rechengraphen des Modells und erzeugt daraus eigene, zusammengefasste GPU-Kernel.
Mehrere kleine Operationen wie LayerNorm, GELU und Additionen laufen dann in einem Schritt, statt jedes Zwischenergebnis in den Speicher zu schreiben.
Das Kompilieren passiert in den ersten Schritten und steckt deshalb nur in der Aufwärmphase, nicht in der Messung.
Folie 20, Punkt 4.

In [ ]:
measure(6)

## 5. Ergebnis

Alle Varianten liefen mit derselben Batchgröße und Sequenzlänge. Steht bei einer Variante „zu wenig Speicher“, hat sie mit diesen Werten nicht auf die GPU gepasst.

In [ ]:
df = table()
if df["Tokens/s"].notna().any():
    df.plot.barh(y="Tokens/s", title="Tokens pro Sekunde (mehr ist besser)", legend=False)
if df["Speicher (GB)"].notna().any():
    df.plot.barh(y="Speicher (GB)", title="Spitzenspeicher in GB (weniger ist besser)", legend=False)
df

## 6. Fazit

Nach dem Lauf auf der GPU ausfüllen: welche GPU, welcher Schritt am meisten bringt, wie hoch die MFU am Ende ist
und wie viel Speicher FlashAttention spart.